In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 1. tensorflow v2.xx에서 v1 사용하기

In [4]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화하고 v1만 활성화
import numpy as np
import pandas as pd

Instructions for updating:
non-resource variables are not supported in the long term


## Tensorflow
- 데이터 흐름 그래프(tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 그래프는 node(데이터, 연산)와 edge로 구성
- sess = tf.Session()을 이용하여 실행환경
- sess.run()을 통해서 실행결과를 확인

In [7]:
# 1. tensor(상수node, 변수node, 연산node) 정의
node1 = tf.constant('Hello, Tensorflow')
# 2. 세션 생성(연산을 실행하는 환경 생성)
sess = tf.Session()
# 3. 실행
print(sess.run(node1))
print(sess.run(node1).decode())

b'Hello, Tensorflow'
Hello, Tensorflow


In [9]:
# 간단한 연산 tensor 그래프
# 1. 그래프 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16)
node3 = tf.add(node1, node2)
# 2. 세션 생성
sess = tf.Session()
# 3. 세션 실행 및 결과
print(sess.run([node1, node2, node3]))

[10.0, 20.0, 30.0]


In [10]:
# 타입 변경
node1 = tf.constant(np.array([10,20,30]), dtype=tf.int16)
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run([node1,node2]))

[array([10, 20, 30], dtype=int16), array([10., 20., 30.], dtype=float32)]


In [12]:
# 평균값 계산 : tf.reduce_mean()
data = np.array([1.,2,3,4])
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

2.5

In [14]:
# tf.random_normal([shape]) : 평균0, 표준편차는 1인 shape 난수 배열tensor. 기본적으로 float32
w = tf.random.normal([1])
sess = tf.Session()
sess.run(w)

array([-0.804195], dtype=float32)

In [17]:
# 변수노드
w = tf.Variable(tf.random.normal([1]))
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 난수가 발생될 변수 초기화
sess.run(w)

array([0.2845762], dtype=float32)

# 2. tensorflow v1을 이용한 회귀분석 구현

## 2.1 독립(입력)변수 x가 1개, 종속(타겟)변수 y가 1개

In [27]:
# tensor 그래프 정의
# 데이터 셋 확보
x = np.array([1,2,3])
y = np.array([2,3,4])
# weight와 bias
w = tf.Variable(tf.random.normal([1]), name='weight')
b = tf.Variable(tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H - y 의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법 : GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train,cost,w,b])
    if step%200==1:
        print(f'{step}번째 cost : {cost_val}, w:{w_val}, b:{b_val}')

1번째 cost : 5.780813217163086, w:[-0.02649003], b:[1.0850029]
201번째 cost : 0.0111244460567832, w:[0.8777947], b:[1.2778013]
401번째 cost : 0.004247842822223902, w:[0.9244847], b:[1.1716639]
601번째 cost : 0.0016220432007685304, w:[0.9533361], b:[1.1060781]
801번째 cost : 0.000619376776739955, w:[0.9711645], b:[1.0655499]
1001번째 cost : 0.00023650069488212466, w:[0.9821816], b:[1.0405052]
1201번째 cost : 9.030706860357895e-05, w:[0.9889893], b:[1.0250298]
1401번째 cost : 3.448418647167273e-05, w:[0.993196], b:[1.015467]
1601번째 cost : 1.3166631106287241e-05, w:[0.99579567], b:[1.0095574]
1801번째 cost : 5.0281305448152125e-06, w:[0.99740195], b:[1.005906]
2001번째 cost : 1.9200103906769073e-06, w:[0.99839455], b:[1.0036496]
2201번째 cost : 7.332460540965258e-07, w:[0.99900776], b:[1.0022554]
2401번째 cost : 2.803230927383993e-07, w:[0.99938655], b:[1.0013944]
2601번째 cost : 1.0714984455262311e-07, w:[0.99962074], b:[1.0008621]
2801번째 cost : 4.1084437185645584e-08, w:[0.9997651], b:[1.0005339]
3001번째 cost : 1

In [28]:
w_, b_ =sess.run([w[0], b[0]])
w_, b_

(0.9999891, 1.0000248)

In [29]:
def predict(x):
    return x*w_ + b_

In [30]:
predict(5)

5.999970257282257

## 2.2 predict을 위한 placeholder이용
- placeholder : 외부에서 데이터를 입력받을 수 있는 노드

In [34]:
x = tf.placeholder(dtype=tf.float32)
H = w_*x + b_
sess = tf.Session()
sess.run([H, x], {x:2.5})

[3.4999976, array(2.5, dtype=float32)]

In [35]:
sess.run(H, {x: np.array([2.5,3,3.5])})

array([3.4999976, 3.9999921, 4.4999866], dtype=float32)

In [36]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,3])
y_data = np.array([2,3,4])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable(tf.random.normal([1]), name='weight')
b = tf.Variable(tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H - y 의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법 : GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train,cost,w,b],
                                         feed_dict={x:x_data, y:y_data})
    if step%200==1:
        print(f'{step}번째 cost : {cost_val}, w:{w_val}, b:{b_val}')

1번째 cost : 23.431480407714844, w:[0.6183611], b:[-2.5566964]
201번째 cost : 0.4437536895275116, w:[1.7718301], b:[-0.7545512]
401번째 cost : 0.1694469004869461, w:[1.4769441], b:[-0.08420596]
601번째 cost : 0.06470312178134918, w:[1.2947226], b:[0.33002642]
801번째 cost : 0.02470678649842739, w:[1.1821206], b:[0.5859973]
1001번째 cost : 0.009434257633984089, w:[1.1125394], b:[0.7441715]
1201번째 cost : 0.0036024637520313263, w:[1.0695425], b:[0.8419135]
1401번째 cost : 0.001375604304485023, w:[1.0429732], b:[0.9023119]
1601번째 cost : 0.0005252670962363482, w:[1.0265547], b:[0.93963486]
1801번째 cost : 0.00020057383517269045, w:[1.0164092], b:[0.962698]
2001번째 cost : 7.658810500288382e-05, w:[1.0101398], b:[0.9769498]
2201번째 cost : 2.9244920369819738e-05, w:[1.0062658], b:[0.98575634]
2401번째 cost : 1.1168030141561758e-05, w:[1.0038722], b:[0.99119794]
2601번째 cost : 4.265050392859848e-06, w:[1.0023928], b:[0.99456054]
2801번째 cost : 1.6290690609821468e-06, w:[1.0014789], b:[0.99663836]
3001번째 cost : 6.223

In [37]:
# 예측하기
sess.run(H, feed_dict={x:2.5})

array([3.5000029], dtype=float32)

In [38]:
sess.run(H, feed_dict={x:np.array([2.5, 3.5])})

array([3.5000029, 4.5000114], dtype=float32)

## 2.3 scale이 다른 데이터들의 회귀분석 구현(scale조정X)

In [40]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,16,68,80,95])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable(tf.random.normal([1]), name='weight')
b = tf.Variable(tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H - y 의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법 : GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 5001):
    _, cost_val, w_val, b_val = sess.run([train,cost,w,b],
                                         feed_dict={x:x_data, y:y_data})
    if step%200==1:
        print(f'{step}번째 cost : {cost_val}, w:{w_val}, b:{b_val}')

1번째 cost : 3977.678955078125, w:[7.879367], b:[1.1474621]
201번째 cost : 77.26811218261719, w:[10.057376], b:[0.60759324]
401번째 cost : 77.23283386230469, w:[10.090883], b:[0.35949737]
601번째 cost : 77.22958374023438, w:[10.101033], b:[0.2843516]
801번째 cost : 77.22928619384766, w:[10.104106], b:[0.26159048]
1001번째 cost : 77.229248046875, w:[10.105038], b:[0.25469637]
1201번째 cost : 77.22926330566406, w:[10.105319], b:[0.25260824]
1401번째 cost : 77.22926330566406, w:[10.105405], b:[0.25197586]
1601번째 cost : 77.22925567626953, w:[10.105431], b:[0.2517848]
1801번째 cost : 77.229248046875, w:[10.105438], b:[0.2517306]
2001번째 cost : 77.22923278808594, w:[10.105441], b:[0.2517113]
2201번째 cost : 77.22926330566406, w:[10.105441], b:[0.25170517]
2401번째 cost : 77.22926330566406, w:[10.105441], b:[0.25170517]
2601번째 cost : 77.22926330566406, w:[10.105441], b:[0.25170517]
2801번째 cost : 77.22926330566406, w:[10.105441], b:[0.25170517]
3001번째 cost : 77.22926330566406, w:[10.105441], b:[0.25170517]
3201번째 co

## 2.4 scale이 다른 데이터의 회귀분석(scale조정O)
### scale조정방법 : 모든데이터를 일정범위내로 조정
- normallization(정규화) : 모든 데이터를 0~1 사이로 조정
                        X - Xmin
    normallization = ───────────────
                       Xmax - Xmin
        * 위의 식보다 라이브러리 추천(sklearn.preprocessing.MinMaxScaler)
- standeardization(표준화) : 데이터의 평균을 0, 표준편차를 1로 조정
                          X - Xmean
    standeardization = ───────────────
                        Xstd(표준편차)
        * * 위의 식보다 라이브러리 추천(sklearn.preprocessing.StandardScaler)

In [44]:
# 라이브러리를 쓰지 않고 정규화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,16,68,80,95])
norm_scaled_x_data = (x_data - x_data.min()) / (x_data.max() - x_data.min())
norm_scaled_y_data = (y_data - y_data.min()) / (y_data.max() - y_data.min())
print(norm_scaled_x_data)
print(norm_scaled_y_data)

[0.         0.11111111 0.44444444 0.77777778 1.        ]
[0.         0.12222222 0.7        0.83333333 1.        ]


In [59]:
# 라이브러리를 사용하요 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler
x_data = np.array([1,2,5,8,10]).reshape(-1, 1)
y_data = np.array([5,15,68,80,95]).reshape(-1, 1)
scaler_x = MinMaxScaler()
scaler_x.fit(x_data)
norm_scaled_x_data = scaler_x.transform(x_data)
scaler_y = MinMaxScaler()
norm_scaled_y_data = scaler_x.fit_transform(y_data)
np.column_stack([x_data, norm_scaled_x_data, y_data, norm_scaled_y_data])

array([[ 1.        ,  0.        ,  5.        ,  0.        ],
       [ 2.        ,  0.11111111, 15.        ,  0.11111111],
       [ 5.        ,  0.44444444, 68.        ,  0.7       ],
       [ 8.        ,  0.77777778, 80.        ,  0.83333333],
       [10.        ,  1.        , 95.        ,  1.        ]])

In [63]:
# 라이브러리를 쓰지 않고 표준화
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,16,68,80,95])
stan_scaled_x_data = ( x_data - x_data.mean() ) / x_data.std()
stan_scaled_y_data = ( y_data - y_data.mean() ) / y_data.std()
print(np.column_stack([x_data, stan_scaled_x_data, norm_scaled_x_data]))
print()
print(np.column_stack([y_data, stan_scaled_y_data, norm_scaled_y_data]))

[[ 1.         -1.22474487  0.        ]
 [ 2.         -0.93313895  0.11111111]
 [ 5.         -0.05832118  0.44444444]
 [ 8.          0.81649658  0.77777778]
 [10.          1.39970842  1.        ]]

[[ 5.         -1.33701194  0.        ]
 [16.         -1.02933137  0.11111111]
 [68.          0.42515861  0.7       ]
 [80.          0.76081014  0.83333333]
 [95.          1.18037456  1.        ]]


In [64]:
# 라이브러리를 사용하여 표준화
x_data = np.array([1,2,5,8,10]).reshape(-1, 1)
y_data = np.array([5,16,68,80,95]).reshape(-1, 1)
scaler_x = StandardScaler()
stan_scaled_x_data = scaler_x.fit_transform(x_data)
scaler_y = StandardScaler()
stan_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack([stan_scaled_x_data, stan_scaled_y_data])

array([[-1.22474487, -1.33701194],
       [-0.93313895, -1.02933137],
       [-0.05832118,  0.42515861],
       [ 0.81649658,  0.76081014],
       [ 1.39970842,  1.18037456]])

In [ ]:
# 스케일 조정된 데이터를 다시 복구 : inverse_transform() 이용
scaler_x.inverse_transform(stan_scaled_x_data)

In [ ]:
scaler_y.inverse_transform(stan_scaled_y_data)

In [ ]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 2001):
    _, cost_val = sess.run([train, cost],
                        feed_dict={x:norm_scaled_x_data,
                                   y:norm_scaled_y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}')
print(f'{step}번째 cost:{cost_val}')

## 2.5 독립변수가 x가 3개, 타겟변수 y가 1개인 회귀분석
- 교안 25p

In [15]:
# data set
X_data = np.array([[73,80,75], # 학습시에는 x에 5행3열 / 예측시 ?행3열
                   [93,88,93],
                   [89,91,90],
                   [96,98,100],
                   [73,66,70]])
y_data = np.array([[152],      # 학습시에는 y에 5행1열 / 예측시에는 ?행1열
                   [185],
                   [180],
                   [196],
                   [142]])
X = tf.placeholder(shape=[None, 3], dtype=tf.float32)
y = tf.placeholder(shape=[None, 1], dtype=tf.float32)
# w & b
W = tf.Variable(tf.random.normal([3, 1]))
b = tf.Variable(tf.random.normal([1]))
# 예측값
# H = X @ W + b @ 행렬곱
H = tf.matmul(X, W) + b
# 손실함수(loss) = : mse
cost = tf.reduce_mean(tf.square(H - y))
# optimizer & train
train = tf.train.GradientDescentOptimizer(learning_rate=0.00004).minimize(cost)
# sess생성 & variable 초기화
sess = tf.Session()
sess.run(tf.global_variables_initializer())
# 학습
for step in range(1, 50001):
    _, cost_val, = sess.run([train, cost], feed_dict={X:X_data, y:y_data})
    if step%5000==1:
        print(f'{step}번째 cost : {cost_val}')
print(f'최종 {step}번째 cost : {cost_val}')

1번째 cost : 54699.33203125
5001번째 cost : 0.32040899991989136
10001번째 cost : 0.22106628119945526
15001번째 cost : 0.18336567282676697
20001번째 cost : 0.16899405419826508
25001번째 cost : 0.16345755755901337
30001번째 cost : 0.16127048432826996
35001번째 cost : 0.16035400331020355
40001번째 cost : 0.15991781651973724
45001번째 cost : 0.15966704487800598
최종 50000번째 cost : 0.15948374569416046


In [17]:
# 예측
input_data = np.array([[72,79,74],
                       [92,87,92]])
sess.run(H, feed_dict={X:input_data})

array([[149.5382 ],
       [182.60654]], dtype=float32)

In [18]:
sess.run(H, feed_dict={X:[[90,85,90]]})

array([[178.58632]], dtype=float32)

# scale조정 후 학습하고 예측하기

In [20]:
from sklearn.preprocessing import StandardScaler
X_data = np.array([[73,80,75], # 학습시에는 x에 5행3열 / 예측시 ?행3열
                   [93,88,93],
                   [89,91,90],
                   [96,98,100],
                   [73,66,70]])
y_data = np.array([[152],      # 학습시에는 y에 5행1열 / 예측시에는 ?행1열
                   [185],
                   [180],
                   [196],
                   [142]])
scaler_x = StandardScaler()
scaler_y = StandardScaler()
scaler_X_data = scaler_x.fit_transform(X_data)
scaler_y_data = scaler_y.fit_transform(y_data)
np.column_stack([scaler_X_data, scaler_y_data])

array([[-1.19344226, -0.42020085, -0.93897274, -0.92622337],
       [ 0.82934123,  0.31058324,  0.65550927,  0.68248038],
       [ 0.42478453,  0.58462728,  0.38976227,  0.43873739],
       [ 1.13275875,  1.22406336,  1.27558561,  1.21871496],
       [-1.19344226, -1.69907302, -1.38188441, -1.41370936]])

In [22]:
X_data = np.array([[73,80,75], # 학습시에는 x에 5행3열 / 예측시 ?행3열
                   [93,88,93],
                   [89,91,90],
                   [96,98,100],
                   [73,66,70]])
y_data = np.array([[152],      # 학습시에는 y에 5행1열 / 예측시에는 ?행1열
                   [185],
                   [180],
                   [196],
                   [142]])
X = tf.placeholder(shape=[None, 3], dtype=tf.float32)
y = tf.placeholder(shape=[None, 1], dtype=tf.float32)
# w & b
W = tf.Variable(tf.random.normal([3, 1]))
b = tf.Variable(tf.random.normal([1]))
# 예측값
# H = X @ W + b @ 행렬곱
H = tf.matmul(X, W) + b
# 손실함수(loss) = : mse
cost = tf.reduce_mean(tf.square(H - y))
# optimizer & train
train = tf.train.GradientDescentOptimizer(learning_rate=0.01).minimize(cost)
# sess생성 & variable 초기화
sess = tf.Session()
sess.run(tf.global_variables_initializer())
# 학습
for step in range(1, 30001):
    _, cost_val, = sess.run([train, cost], feed_dict={X:scaler_X_data, y:scaler_y_data})
    if step%3000==1:
        print(f'{step}번째 cost : {cost_val}')
print(f'최종 {step}번째 cost : {cost_val}')

1번째 cost : 2.2757015228271484
3001번째 cost : 0.0005480502732098103
6001번째 cost : 0.0004768619255628437
9001번째 cost : 0.0004225804877933115
12001번째 cost : 0.0003811693168245256
15001번째 cost : 0.00034957827301695943
18001번째 cost : 0.00032547698356211185
21001번째 cost : 0.00030709218117408454
24001번째 cost : 0.000293064396828413
27001번째 cost : 0.0002823631220962852
최종 30000번째 cost : 0.00027420115657150745


In [23]:
# 예측
input_data = np.array([[72,79,74],
                       [92,87,92]])
scaled_input_data = scaler_x.transform(input_data)
print('scale조정된 입력값 : \n',scaled_input_data)
hat = sess.run(H, feed_dict={X : scaled_input_data})
predict_data = scaler_y.inverse_transform(hat)
print('예측 점수 :\n', predict_data)

scale조정된 입력값 : 
 [[-1.29458143 -0.51154887 -1.02755507]
 [ 0.72820205  0.21923523  0.56692694]]
예측 점수 :
 [[149.72562]
 [182.54388]]


In [24]:
input_data = [[90,85,90]]
hat = sess.run(H, feed_dict={X : scaler_x.transform(input_data)})
scaler_y.inverse_transform(hat)

array([[178.58534]], dtype=float32)